# LCEL 체인 예제 - Gemini API 버전

이 노트북은 Google Gemini API를 사용하여 LangChain Expression Language (LCEL)의 다양한 체인 기능을 보여줍니다.

## 주요 학습 내용:
- 기본 메시지와 모델 호출
- 출력 파서(Output Parser) 사용법
- 체인(Chain) 구성과 파이프라인
- 프롬프트 템플릿 활용
- 구조화된 출력 생성

## 시나리오:
미녀와 야수 스토리를 배경으로 한 캐릭터 대화 시뮬레이션

## 1. 환경 설정 및 모델 초기화

In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from typing import Literal
from pydantic import BaseModel, Field

In [ ]:
# 환경변수 로드
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise ValueError("환경변수 GOOGLE_API_KEY가 설정되지 않았습니다.")

print("✅ 환경변수가 성공적으로 로드되었습니다.")

In [ ]:
# Gemini 모델 초기화
model = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0.7,
)

print("✅ Gemini 모델이 초기화되었습니다.")

## 2. 기본 메시지와 모델 호출

In [ ]:
# 시스템 메시지와 사용자 메시지 구성
messages = [
    SystemMessage(content="너는 미녀와 야수에 나오는 미녀야. 그 캐릭터에 맞게 사용자와 대화하라."),
    HumanMessage(content="안녕? 저는 개스톤입니다. 오늘 시간 괜찮으시면 저녁 같이 먹을까요?"),
]

# 모델 호출
result = model.invoke(messages)
print("🤖 AI 응답:")
print(result.content)
print(f"\n📋 응답 타입: {type(result)}")

## 3. 출력 파서(Output Parser) 사용

In [ ]:
# 출력 파서 생성
parser = StrOutputParser()

# 같은 메시지로 모델 호출
messages = [
    SystemMessage(content="너는 미녀와 야수에 나오는 미녀야. 그 캐릭터에 맞게 사용자와 대화하라."),
    HumanMessage(content="안녕? 저는 개스톤입니다. 오늘 시간 괜찮으시면 저녁 같이 먹을까요?"),
]

result = model.invoke(messages)  # AIMessage 객체
parsed_result = parser.invoke(result)  # 문자열로 파싱

print("🤖 파싱된 응답:")
print(parsed_result)
print(f"\n📋 파싱된 타입: {type(parsed_result)}")

## 4. 기본 체인 구성 (모델 | 파서)

In [ ]:
# 파이프라인 연산자(|)를 사용한 체인 구성
parser = StrOutputParser()
chain = model | parser

# 체인 실행
messages = [
    SystemMessage(content="너는 미녀와 야수에 나오는 미녀야. 그 캐릭터에 맞게 사용자와 대화하라."),
    HumanMessage(content="안녕? 저는 개스톤입니다. 오늘 시간 괜찮으시면 저녁 같이 먹을까요?"),
]

result = chain.invoke(messages)  # 직접 문자열 반환
print("🤖 체인 응답:")
print(result)
print(f"\n📋 결과 타입: {type(result)}")

## 5. 프롬프트 템플릿 사용

In [ ]:
# 프롬프트 템플릿 정의
system_template = "너는 {story}에 나오는 {character_a} 역할이다. 그 캐릭터에 맞게 사용자와 대화하라."
human_template = "안녕? 저는 {character_b}입니다. 오늘 시간 괜찮으시면 {activity} 같이 할까요?"

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_template),
    ("user", human_template),
])

# 프롬프트 템플릿 테스트
prompt_value = prompt_template.invoke({
    "story": "미녀와 야수",
    "character_a": "미녀",
    "character_b": "야수",
    "activity": "저녁"
})

print("📋 생성된 메시지들:")
for message in prompt_value.to_messages():
    print(f"  - {message.__class__.__name__}: {message.content}")

## 6. 완전한 체인 구성 (프롬프트 | 모델 | 파서)

In [ ]:
# 완전한 체인 구성
system_template = "너는 {story}에 나오는 {character_a} 역할이다. 그 캐릭터에 맞게 사용자와 대화하라."
human_template = "안녕? 저는 {character_b}입니다. 오늘 시간 괜찮으시면 {activity} 같이 할까요?"

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_template),
    ("user", human_template),
])

parser = StrOutputParser()
chain = prompt_template | model | parser

print("✅ 완전한 체인이 구성되었습니다: prompt_template | model | parser")

In [ ]:
# 테스트 1: 야수와의 대화
print("🌹 야수와의 대화:")
print("=" * 30)

result1 = chain.invoke({
    "story": "미녀와 야수",
    "character_a": "미녀",
    "character_b": "야수",
    "activity": "저녁"
})

print(f"🤖 미녀의 응답: {result1}")

In [ ]:
# 테스트 2: 개스톤과의 대화
print("💪 개스톤과의 대화:")
print("=" * 30)

result2 = chain.invoke({
    "story": "미녀와 야수",
    "character_a": "미녀",
    "character_b": "개스톤",
    "activity": "저녁"
})

print(f"🤖 미녀의 응답: {result2}")

## 7. 구조화된 출력 (Pydantic 모델)

In [ ]:
# Pydantic 모델 정의
class Adlib(BaseModel):
    """스토리 설정과 사용자 입력에 반응하는 대사를 만드는 클래스"""
    answer: str = Field(description="스토리 설정과 사용자와의 대화 기록에 따라 생성된 대사")
    main_emotion: Literal["기쁨", "분노", "슬픔", "공포", "냉소", "불쾌", "중립"] = Field(description="대사의 주요 감정")
    main_emotion_intensity: float = Field(description="대사의 주요 감정의 강도 (0.0 ~ 1.0)")

print("📋 Adlib 모델이 정의되었습니다.")
print(f"   - 필드: answer, main_emotion, main_emotion_intensity")
print(f"   - 감정 옵션: {Adlib.__annotations__['main_emotion'].__args__}")

In [ ]:
# 구조화된 출력을 위한 체인 구성
system_template = "너는 {story}에 나오는 {character_a} 역할이다. 그 캐릭터에 맞게 사용자와 대화하라."
human_template = "안녕? 저는 {character_b}입니다. 오늘 시간 괜찮으시면 {activity} 같이 할까요?"

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_template),
    ("user", human_template),
])

try:
    # LangChain이 Gemini의 response schema를 활용하여 JSON 구조로 강제
    structured_llm = model.with_structured_output(Adlib)
    adlib_chain = prompt_template | structured_llm
    
    print("✅ 구조화된 출력 체인이 구성되었습니다.")
    
except Exception as e:
    print(f"⚠️ 구조화된 출력 설정 실패: {e}")
    print("💡 일반 체인을 사용하겠습니다.")

In [ ]:
# 구조화된 출력 테스트
try:
    result = adlib_chain.invoke({
        "story": "미녀와 야수",
        "character_a": "벨",
        "character_b": "개스톤",
        "activity": "저녁"
    })
    
    print("🤖 구조화된 응답:")
    print("=" * 40)
    print(f"💬 대사: {result.answer}")
    print(f"😊 감정: {result.main_emotion}")
    print(f"📊 강도: {result.main_emotion_intensity}")
    print(f"\n📋 결과 타입: {type(result)}")
    
except Exception as e:
    print(f"⚠️ 구조화된 출력 지원 안됨: {e}")
    print("💡 일반 체인으로 대체 실행:")
    
    # 폴백: 일반 체인 사용
    parser = StrOutputParser()
    fallback_chain = prompt_template | model | parser
    
    result = fallback_chain.invoke({
        "story": "미녀와 야수",
        "character_a": "벨",
        "character_b": "개스톤",
        "activity": "저녁"
    })
    print(f"🤖 일반 응답: {result}")

## 8. 다양한 시나리오 테스트

In [ ]:
# 다양한 캐릭터와 활동으로 테스트
test_scenarios = [
    {
        "story": "미녀와 야수",
        "character_a": "미녀",
        "character_b": "왕자",
        "activity": "산책",
        "emoji": "👑"
    },
    {
        "story": "미녀와 야수",
        "character_a": "미녀",
        "character_b": "마법사",
        "activity": "도서관 구경",
        "emoji": "🧙‍♂️"
    },
    {
        "story": "미녀와 야수",
        "character_a": "미녀",
        "character_b": "촛대",
        "activity": "차 한잔",
        "emoji": "🕯️"
    }
]

# 기본 체인 사용
chain = prompt_template | model | parser

print("🎭 다양한 시나리오 테스트")
print("=" * 50)

for i, scenario in enumerate(test_scenarios, 1):
    print(f"\n{scenario['emoji']} 테스트 {i}: {scenario['character_b']}와의 {scenario['activity']}")
    print("-" * 30)
    
    result = chain.invoke(scenario)
    print(f"🤖 미녀의 응답: {result}")

## 9. 체인 성능 비교

In [ ]:
import time

# 테스트 데이터
test_data = {
    "story": "미녀와 야수",
    "character_a": "미녀",
    "character_b": "개스톤",
    "activity": "저녁"
}

print("⚡ 체인 성능 비교")
print("=" * 40)

# 방법 1: 개별 호출
start_time = time.time()
prompt_value = prompt_template.invoke(test_data)
messages = prompt_value.to_messages()
result = model.invoke(messages)
parsed_result = parser.invoke(result)
method1_time = time.time() - start_time

print(f"📊 방법 1 (개별 호출): {method1_time:.3f}초")

# 방법 2: 체인 사용
start_time = time.time()
chain_result = chain.invoke(test_data)
method2_time = time.time() - start_time

print(f"📊 방법 2 (체인 사용): {method2_time:.3f}초")

# 결과 비교
print(f"\n📋 결과 비교:")
print(f"  - 방법 1 결과 길이: {len(parsed_result)}자")
print(f"  - 방법 2 결과 길이: {len(chain_result)}자")
print(f"  - 성능 차이: {abs(method1_time - method2_time):.3f}초")

## 10. 요약 및 학습 정리

이 노트북에서 학습한 LCEL 체인의 주요 개념들을 정리해보겠습니다.

In [ ]:
print("🎉 LCEL 체인 예제 - Gemini API 버전 완료!")
print("=" * 50)

print("\n📚 학습한 주요 개념:")
print("  1️⃣ 기본 메시지 구성 (SystemMessage, HumanMessage)")
print("  2️⃣ 출력 파서 (StrOutputParser)로 결과 정리")
print("  3️⃣ 파이프라인 연산자(|)를 사용한 체인 구성")
print("  4️⃣ 프롬프트 템플릿으로 동적 프롬프트 생성")
print("  5️⃣ 완전한 체인 (프롬프트 | 모델 | 파서)")
print("  6️⃣ Pydantic 모델을 활용한 구조화된 출력")

print("\n🔧 사용한 주요 도구:")
print("  - Google Gemini 2.0 Flash 모델")
print("  - LangChain Expression Language (LCEL)")
print("  - ChatPromptTemplate")
print("  - StrOutputParser")
print("  - Pydantic BaseModel")

print("\n💡 핵심 포인트:")
print("  ✅ 체인은 파이프라인 연산자(|)로 간단하게 연결")
print("  ✅ 프롬프트 템플릿으로 재사용 가능한 대화 패턴 구성")
print("  ✅ 구조화된 출력으로 일관된 응답 형식 보장")
print("  ✅ 각 단계를 모듈화하여 유지보수성 향상")

print("\n🚀 다음 단계:")
print("  - 메시지 히스토리와 체인 결합")
print("  - 조건부 체인과 분기 처리")
print("  - 병렬 체인과 결과 합성")
print("  - 사용자 정의 출력 파서 개발")